In [1]:
import os
# Enforce single GPU execution to prevent multi-GPU synchronization conflicts
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import math
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from safetensors.torch import load_file
from huggingface_hub import login, hf_hub_download

# Check for bitsandbytes availability
try:
    import bitsandbytes as bnb
    HAS_BNB = True
    print("bitsandbytes loaded successfully. 4-bit NF4 QLoRA quantization enabled.")
except ImportError:
    HAS_BNB = False
    print("bitsandbytes not found. Falling back to standard frozen-weight LoRA.")

# ==============================================================================
# GLOBAL DEBUG CONTROL
# Set DEBUG = True for rapid code verification, False for full production run.
# ==============================================================================
DEBUG = False

DEBUG_CONFIG = {
    "train_samples": 64,
    "dev_samples": 16,
    "test_samples": 8,
    "epochs": 1,
    "batch_size": 2,
    "log_interval": 2,
    "lr": 3e-4,
    "qlora_rank": 16,
    "qlora_alpha": 32,
    "max_doc_len": 768,
    "max_summ_len": 256,
    "gen_max_new_tokens": 300,
    "gen_min_new_tokens": 120
}

FULL_CONFIG = {
    "train_samples": None,  # Full dataset
    "dev_samples": None,
    "test_samples": None,
    "epochs": 3,
    "batch_size": 2,
    "log_interval": 20,
    "lr": 3e-4,
    "qlora_rank": 16,
    "qlora_alpha": 32,
    "max_doc_len": 1536,
    "max_summ_len": 512,
    "gen_max_new_tokens": 400,
    "gen_min_new_tokens": 150
}

CFG = DEBUG_CONFIG if DEBUG else FULL_CONFIG

print(f"=== Phase 6 Execution Mode: {'DEBUG (Fast Verification Run)' if DEBUG else 'PRODUCTION RUN'} ===")
print(f"Configuration: {CFG}\n")

bitsandbytes not found. Falling back to standard frozen-weight LoRA.
=== Phase 6 Execution Mode: PRODUCTION RUN ===
Configuration: {'train_samples': None, 'dev_samples': None, 'test_samples': None, 'epochs': 3, 'batch_size': 2, 'log_interval': 20, 'lr': 0.0003, 'qlora_rank': 16, 'qlora_alpha': 32, 'max_doc_len': 1536, 'max_summ_len': 512, 'gen_max_new_tokens': 400, 'gen_min_new_tokens': 150}



In [ ]:
HF_TOKEN = os.environ["HF_TOKEN"]
login(token=HF_TOKEN)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MODEL_REPO_ENC_DEC = "A-240242/legal-encdec-checkpoints" 
HF_TOKENIZER_REPO = "A-240242/legal-modern-decoder-checkpoints"

def load_summ_file(filename):
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO, 
        filename=f"Phase_6_Summ/{filename}", 
        repo_type="dataset", 
        token=HF_TOKEN
    )
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("Downloading Phase 6 Abstractive Summarization Data...")
full_train_data = load_summ_file("Train/train.json")
test_data = load_summ_file("Test/test.json")

print(f"Loaded {len(full_train_data)} total training examples and {len(test_data)} test examples.")

# Construct a held-out validation split (10%) from train.json
np.random.seed(42)
indices = np.arange(len(full_train_data))
np.random.shuffle(indices)

val_size = max(1, int(0.10 * len(full_train_data)))
val_indices = set(indices[:val_size])

train_data = [full_train_data[i] for i in range(len(full_train_data)) if i not in val_indices]
dev_data = [full_train_data[i] for i in val_indices]

print(f"Split data into {len(train_data)} training samples and {len(dev_data)} validation samples.")

if DEBUG and CFG["train_samples"]:
    train_data = train_data[:CFG["train_samples"]]
    dev_data = dev_data[:CFG["dev_samples"]]
    test_data = test_data[:CFG["test_samples"]]
    print(f"\n[DEBUG] Subsampled train to {len(train_data)}, dev to {len(dev_data)}, and test to {len(test_data)} records.")

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

Loaded 7030 total training examples and 100 test examples.
Split data into 6327 training samples and 703 validation samples.


In [3]:
tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER_REPO, token=HF_TOKEN)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

class Seq2SeqSummarizationDataset(Dataset):
    """
    Sequence-to-Sequence Dataset for Encoder-Decoder Abstractive Summarization.
    Encoder Inputs: Tokenized legal document text.
    Decoder Inputs/Labels: Shifted target summary token sequence with -100 padding mask.
    """
    def __init__(self, raw_records, tokenizer, max_doc_len=1536, max_summ_len=512, is_test=False):
        self.samples = []
        self.is_test = is_test
        
        for record in raw_records:
            doc_id = record.get("id")
            doc_text = record.get("document", "")
            
            enc_ids = tokenizer.encode(doc_text, truncation=True, max_length=max_doc_len)
            
            if is_test:
                self.samples.append({
                    "id": doc_id,
                    "encoder_input_ids": enc_ids
                })
            else:
                summ_text = record.get("summary", "")
                dec_ids = tokenizer.encode(summ_text, truncation=True, max_length=max_summ_len)
                if tokenizer.eos_token_id not in dec_ids:
                    dec_ids.append(tokenizer.eos_token_id)
                    
                self.samples.append({
                    "id": doc_id,
                    "encoder_input_ids": enc_ids,
                    "decoder_input_ids": [tokenizer.pad_token_id] + dec_ids[:-1],
                    "labels": dec_ids
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def seq2seq_collate_fn(batch):
    if "labels" in batch[0]:
        enc_input_ids = [torch.tensor(item["encoder_input_ids"], dtype=torch.long) for item in batch]
        dec_input_ids = [torch.tensor(item["decoder_input_ids"], dtype=torch.long) for item in batch]
        labels_list = [torch.tensor(item["labels"], dtype=torch.long) for item in batch]
        doc_ids = [item["id"] for item in batch]
        
        padded_enc_inputs = torch.nn.utils.rnn.pad_sequence(
            enc_input_ids, batch_first=True, padding_value=tokenizer.pad_token_id
        )
        padded_dec_inputs = torch.nn.utils.rnn.pad_sequence(
            dec_input_ids, batch_first=True, padding_value=tokenizer.pad_token_id
        )
        padded_labels = torch.nn.utils.rnn.pad_sequence(
            labels_list, batch_first=True, padding_value=-100
        )
        
        enc_mask = (padded_enc_inputs != tokenizer.pad_token_id).long()
        dec_mask = (padded_dec_inputs != tokenizer.pad_token_id).long()
        
        return {
            "id": doc_ids,
            "encoder_input_ids": padded_enc_inputs,
            "encoder_attention_mask": enc_mask,
            "decoder_input_ids": padded_dec_inputs,
            "decoder_attention_mask": dec_mask,
            "labels": padded_labels
        }
    else:
        enc_input_ids = [torch.tensor(item["encoder_input_ids"], dtype=torch.long) for item in batch]
        doc_ids = [item["id"] for item in batch]
        
        padded_enc_inputs = torch.nn.utils.rnn.pad_sequence(
            enc_input_ids, batch_first=True, padding_value=tokenizer.pad_token_id
        )
        enc_mask = (padded_enc_inputs != tokenizer.pad_token_id).long()
        
        return {
            "id": doc_ids,
            "encoder_input_ids": padded_enc_inputs,
            "encoder_attention_mask": enc_mask
        }

print("Building Phase 6 Seq2Seq DataLoaders...")
train_dataset = Seq2SeqSummarizationDataset(train_data, tokenizer, max_doc_len=CFG["max_doc_len"], max_summ_len=CFG["max_summ_len"])
dev_dataset = Seq2SeqSummarizationDataset(dev_data, tokenizer, max_doc_len=CFG["max_doc_len"], max_summ_len=CFG["max_summ_len"])

train_loader = DataLoader(train_dataset, batch_size=CFG["batch_size"], shuffle=True, collate_fn=seq2seq_collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=CFG["batch_size"], shuffle=False, collate_fn=seq2seq_collate_fn)

print(f"Created {len(train_dataset)} train samples and {len(dev_dataset)} dev samples.")

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

[transformers] You are using a model of type `modern_decoder` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


tokenizer_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

Building Phase 6 Seq2Seq DataLoaders...
Created 6327 train samples and 703 dev samples.


In [4]:
class LayerNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias = nn.Parameter(torch.zeros(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(-1, keepdim=True)
        variance = x.var(-1, keepdim=True, unbiased=False)
        return (x - mean) / torch.sqrt(variance + self.eps) * self.weight + self.bias


class MultiHeadAttention(nn.Module):
    """Multi-Head Self & Cross Attention Module."""
    def __init__(self, hidden_size=768, num_heads=12, is_causal=False):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.is_causal = is_causal
        
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor, key_value_states: torch.Tensor = None, attention_mask: torch.Tensor = None):
        B, S, _ = x.shape
        is_cross = key_value_states is not None
        kv_states = key_value_states if is_cross else x
        
        q = self.q_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(kv_states).view(B, kv_states.shape[1], self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(kv_states).view(B, kv_states.shape[1], self.num_heads, self.head_dim).transpose(1, 2)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        if self.is_causal and not is_cross:
            causal_mask = torch.full((S, S), float("-inf"), device=x.device)
            causal_mask = torch.triu(causal_mask, diagonal=1)
            scores = scores + causal_mask[None, None, :, :]
            
        if attention_mask is not None:
            padding_mask = (1.0 - attention_mask[:, None, None, :].float()) * -10000.0
            scores = scores + padding_mask
            
        probs = F.softmax(scores, dim=-1)
        attn_out = torch.matmul(probs, v).transpose(1, 2).contiguous().view(B, S, self.hidden_size)
        return self.o_proj(attn_out)


class PositionwiseFFN(nn.Module):
    """SwiGLU Feed-Forward Network."""
    def __init__(self, hidden_size=768, intermediate_size=2048):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class EncoderLayer(nn.Module):
    def __init__(self, hidden_size=768, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.input_layernorm = LayerNorm(hidden_size)
        self.self_attn = MultiHeadAttention(hidden_size, num_heads, is_causal=False)
        self.post_attention_layernorm = LayerNorm(hidden_size)
        self.mlp = PositionwiseFFN(hidden_size, intermediate_size)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None):
        residual = x
        x = self.self_attn(self.input_layernorm(x), attention_mask=attention_mask)
        x = residual + x
        
        residual = x
        x = self.mlp(self.post_attention_layernorm(x))
        x = residual + x
        return x


class DecoderLayer(nn.Module):
    def __init__(self, hidden_size=768, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.input_layernorm = LayerNorm(hidden_size)
        self.self_attn = MultiHeadAttention(hidden_size, num_heads, is_causal=True)
        self.cross_attn_layernorm = LayerNorm(hidden_size)
        self.encoder_attn = MultiHeadAttention(hidden_size, num_heads, is_causal=False)
        self.post_attention_layernorm = LayerNorm(hidden_size)
        self.mlp = PositionwiseFFN(hidden_size, intermediate_size)

    def forward(self, x: torch.Tensor, encoder_hidden_states: torch.Tensor, encoder_attention_mask: torch.Tensor = None):
        residual = x
        x = self.self_attn(self.input_layernorm(x))
        x = residual + x
        
        residual = x
        x = self.encoder_attn(
            self.cross_attn_layernorm(x), 
            key_value_states=encoder_hidden_states, 
            attention_mask=encoder_attention_mask
        )
        x = residual + x
        
        residual = x
        x = self.mlp(self.post_attention_layernorm(x))
        x = residual + x
        return x


class CustomEncoderDecoderBackbone(nn.Module):
    """Transformer Encoder-Decoder Backbone."""
    def __init__(self, vocab_size, hidden_size=768, num_encoder_layers=6, num_decoder_layers=6, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.embed_tokens = nn.Embedding(vocab_size, hidden_size)
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(hidden_size, num_heads, intermediate_size) for _ in range(num_encoder_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(hidden_size, num_heads, intermediate_size) for _ in range(num_decoder_layers)
        ])
        self.encoder_norm = LayerNorm(hidden_size)
        self.decoder_norm = LayerNorm(hidden_size)

    def encode(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None):
        x = self.embed_tokens(input_ids)
        for layer in self.encoder_layers:
            x = layer(x, attention_mask=attention_mask)
        return self.encoder_norm(x)

    def decode(self, decoder_input_ids: torch.Tensor, encoder_hidden_states: torch.Tensor, encoder_attention_mask: torch.Tensor = None):
        x = self.embed_tokens(decoder_input_ids)
        for layer in self.decoder_layers:
            x = layer(x, encoder_hidden_states=encoder_hidden_states, encoder_attention_mask=encoder_attention_mask)
        return self.decoder_norm(x)


class CustomEncoderDecoderForConditionalGeneration(nn.Module):
    """Seq2Seq Encoder-Decoder Model for Abstractive Summarization."""
    def __init__(self, vocab_size, hidden_size=768, num_encoder_layers=6, num_decoder_layers=6, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.model = CustomEncoderDecoderBackbone(
            vocab_size, hidden_size, num_encoder_layers, num_decoder_layers, num_heads, intermediate_size
        )
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)

    def forward(
        self, 
        encoder_input_ids: torch.Tensor, 
        encoder_attention_mask: torch.Tensor = None, 
        decoder_input_ids: torch.Tensor = None,
        labels: torch.Tensor = None
    ):
        encoder_hidden_states = self.model.encode(
            input_ids=encoder_input_ids, 
            attention_mask=encoder_attention_mask
        )
        
        decoder_hidden_states = self.model.decode(
            decoder_input_ids=decoder_input_ids,
            encoder_hidden_states=encoder_hidden_states,
            encoder_attention_mask=encoder_attention_mask
        )
        
        logits = self.lm_head(decoder_hidden_states)
        
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
            
        return {"loss": loss, "logits": logits}

print("Custom Encoder-Decoder Architecture written from scratch successfully.")

Custom Encoder-Decoder Architecture written from scratch successfully.


In [5]:
class LoRAAdapter(nn.Module):
    """Trainable Low-Rank Adaptation Matrix Pair in FP32."""
    def __init__(self, in_features, out_features, rank=16, alpha=32):
        super().__init__()
        self.scaling = alpha / rank
        self.lora_A = nn.Linear(in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        return self.scaling * self.lora_B(self.lora_A(x))


class QLoRAWrappedLinear(nn.Module):
    """
    Wraps a base Linear layer with 4-bit NormalFloat (NF4) quantization 
    and a trainable FP32 LoRA adapter.
    """
    def __init__(self, base_linear: nn.Linear, rank=16, alpha=32, quant_type="nf4"):
        super().__init__()
        in_feat = base_linear.in_features
        out_feat = base_linear.out_features
        has_bias = base_linear.bias is not None

        if HAS_BNB:
            self.base = bnb.nn.Linear4bit(
                in_feat,
                out_feat,
                bias=has_bias,
                compute_dtype=torch.float32,
                quant_type=quant_type
            )
            self.base.weight = bnb.nn.Params4bit(
                base_linear.weight.data.clone(),
                requires_grad=False,
                quant_type=quant_type
            )
            if has_bias:
                self.base.bias = nn.Parameter(base_linear.bias.data.clone(), requires_grad=False)
        else:
            self.base = base_linear
            self.base.weight.requires_grad = False
            if self.base.bias is not None:
                self.base.bias.requires_grad = False

        self.adapter = LoRAAdapter(in_feat, out_feat, rank=rank, alpha=alpha)

    def forward(self, x):
        return self.base(x) + self.adapter(x)


# 1. Instantiate Full-Precision Model First
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomEncoderDecoderForConditionalGeneration(
    vocab_size=len(tokenizer),
    hidden_size=768,
    num_encoder_layers=6,
    num_decoder_layers=6,
    num_heads=12,
    intermediate_size=2048
)

# 2. Fetch and Load Phase 1 Pre-trained Checkpoint Weights
print(f"Fetching Phase 1 Checkpoint Weights from '{HF_MODEL_REPO_ENC_DEC}'...")
try:
    weight_path = hf_hub_download(repo_id=HF_MODEL_REPO_ENC_DEC, filename="model.safetensors", token=HF_TOKEN)
    state_dict = load_file(weight_path)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print("Loaded Phase 1 weights successfully into Encoder-Decoder model.")
except Exception as e:
    print(f"Note on weight loading: {e}. Initializing model structure for pipeline execution.")

# 3. Quantize Base Weights to 4-bit NF4 & Inject QLoRA Adapters across Encoder & Decoder
print("Quantizing Base Linear Projections to 4-bit NF4 & Injecting QLoRA Adapters...")
for param in model.parameters():
    param.requires_grad = False

# Unfreeze LM head
for param in model.lm_head.parameters():
    param.requires_grad = True

# Wrap Encoder attention and MLP projections
for layer in model.model.encoder_layers:
    for target_attr in ["q_proj", "k_proj", "v_proj", "o_proj"]:
        base_linear = getattr(layer.self_attn, target_attr)
        setattr(layer.self_attn, target_attr, QLoRAWrappedLinear(base_linear, rank=CFG["qlora_rank"], alpha=CFG["qlora_alpha"]))
        
    for target_attr in ["gate_proj", "up_proj", "down_proj"]:
        base_linear = getattr(layer.mlp, target_attr)
        setattr(layer.mlp, target_attr, QLoRAWrappedLinear(base_linear, rank=CFG["qlora_rank"], alpha=CFG["qlora_alpha"]))

# Wrap Decoder self-attention, cross-attention, and MLP projections
for layer in model.model.decoder_layers:
    for target_attr in ["q_proj", "k_proj", "v_proj", "o_proj"]:
        base_linear = getattr(layer.self_attn, target_attr)
        setattr(layer.self_attn, target_attr, QLoRAWrappedLinear(base_linear, rank=CFG["qlora_rank"], alpha=CFG["qlora_alpha"]))
        
        base_linear_cross = getattr(layer.encoder_attn, target_attr)
        setattr(layer.encoder_attn, target_attr, QLoRAWrappedLinear(base_linear_cross, rank=CFG["qlora_rank"], alpha=CFG["qlora_alpha"]))
        
    for target_attr in ["gate_proj", "up_proj", "down_proj"]:
        base_linear = getattr(layer.mlp, target_attr)
        setattr(layer.mlp, target_attr, QLoRAWrappedLinear(base_linear, rank=CFG["qlora_rank"], alpha=CFG["qlora_alpha"]))

# Ensure trainable adapter parameters are in FP32
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nPhase 6 Encoder-Decoder QLoRA Readiness Summary:")
print(f"  - Total Parameters: {total_params:,}")
print(f"  - Trainable Parameters (QLoRA + Head): {trainable_params:,}")
print(f"  - Frozen 4-bit Base Parameters: {total_params - trainable_params:,}")

Fetching Phase 1 Checkpoint Weights from 'A-240242/legal-encdec-checkpoints'...


model.safetensors: reconstructing file:   0%|          |  0.00B /  521MB            

model.safetensors: downloading bytes:           |  0.00B            

Loaded Phase 1 weights successfully into Encoder-Decoder model.
Quantizing Base Linear Projections to 4-bit NF4 & Injecting QLoRA Adapters...

Phase 6 Encoder-Decoder QLoRA Readiness Summary:
  - Total Parameters: 151,683,072
  - Trainable Parameters (QLoRA + Head): 27,967,488
  - Frozen 4-bit Base Parameters: 123,715,584


In [6]:
CHECKPOINT_DIR = "./checkpoints_phase6_enc_dec_qlora"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BEST_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "best_enc_dec_qlora_summ.pt")

epochs = CFG["epochs"]
lr = CFG["lr"]

trainable_param_list = [p for p in model.parameters() if p.requires_grad]

if HAS_BNB:
    optimizer = bnb.optim.PagedAdamW8bit(trainable_param_list, lr=lr, weight_decay=0.01)
else:
    optimizer = torch.optim.AdamW(trainable_param_list, lr=lr, weight_decay=0.01)

best_val_loss = float("inf")

print(f"\nCommencing Encoder-Decoder QLoRA Summarization Training ({epochs} Epochs)...")

for epoch in range(epochs):
    # --------------------------------------------------------------------------
    # 1. Training Phase
    # --------------------------------------------------------------------------
    model.train()
    total_train_loss = 0.0
    
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        
        enc_ids = batch["encoder_input_ids"].to(device)
        enc_mask = batch["encoder_attention_mask"].to(device)
        dec_ids = batch["decoder_input_ids"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(
            encoder_input_ids=enc_ids, 
            encoder_attention_mask=enc_mask, 
            decoder_input_ids=dec_ids, 
            labels=labels
        )
        loss = outputs["loss"]
        
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        
        if (step + 1) % CFG["log_interval"] == 0 or (step + 1) == len(train_loader):
            print(f"Epoch {epoch+1}/{epochs} | Step {step+1}/{len(train_loader)} | Train Loss: {loss.item():.4f}")
            
    avg_train_loss = total_train_loss / len(train_loader)
    
    # --------------------------------------------------------------------------
    # 2. Validation Phase
    # --------------------------------------------------------------------------
    model.eval()
    total_val_loss = 0.0

    with torch.no_grad():
        for batch in dev_loader:
            enc_ids = batch["encoder_input_ids"].to(device)
            enc_mask = batch["encoder_attention_mask"].to(device)
            dec_ids = batch["decoder_input_ids"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = model(
                encoder_input_ids=enc_ids, 
                encoder_attention_mask=enc_mask, 
                decoder_input_ids=dec_ids, 
                labels=labels
            )
            total_val_loss += outputs["loss"].item()

    avg_val_loss = total_val_loss / len(dev_loader)
    print(f"\nEpoch {epoch+1} Summary | Train Loss: {avg_train_loss:.4f} | Validation Loss: {avg_val_loss:.4f}")

    # --------------------------------------------------------------------------
    # 3. Checkpoint Saving
    # --------------------------------------------------------------------------
    if avg_val_loss < best_val_loss or epoch == 0:
        best_val_loss = avg_val_loss
        
        trainable_state_dict = {
            k: v.cpu().clone() for k, v in model.state_dict().items() if v.requires_grad
        }
        
        checkpoint_data = {
            "epoch": epoch + 1,
            "best_val_loss": best_val_loss,
            "trainable_state_dict": trainable_state_dict
        }
        
        torch.save(checkpoint_data, BEST_CHECKPOINT_PATH)
        print(f"  [+] Saved New Best Checkpoint to '{BEST_CHECKPOINT_PATH}' (Val Loss: {best_val_loss:.4f})")
        
    print("-" * 75)


Commencing Encoder-Decoder QLoRA Summarization Training (3 Epochs)...
Epoch 1/3 | Step 20/3164 | Train Loss: 60.8873
Epoch 1/3 | Step 40/3164 | Train Loss: 25.1789
Epoch 1/3 | Step 60/3164 | Train Loss: 15.8785
Epoch 1/3 | Step 80/3164 | Train Loss: 14.4195
Epoch 1/3 | Step 100/3164 | Train Loss: 14.9589
Epoch 1/3 | Step 120/3164 | Train Loss: 11.3266
Epoch 1/3 | Step 140/3164 | Train Loss: 9.5818
Epoch 1/3 | Step 160/3164 | Train Loss: 8.5856
Epoch 1/3 | Step 180/3164 | Train Loss: 8.6010
Epoch 1/3 | Step 200/3164 | Train Loss: 8.6190
Epoch 1/3 | Step 220/3164 | Train Loss: 9.5172
Epoch 1/3 | Step 240/3164 | Train Loss: 8.7824
Epoch 1/3 | Step 260/3164 | Train Loss: 8.1147
Epoch 1/3 | Step 280/3164 | Train Loss: 8.8156
Epoch 1/3 | Step 300/3164 | Train Loss: 7.9428
Epoch 1/3 | Step 320/3164 | Train Loss: 7.2804
Epoch 1/3 | Step 340/3164 | Train Loss: 8.1354
Epoch 1/3 | Step 360/3164 | Train Loss: 7.5722
Epoch 1/3 | Step 380/3164 | Train Loss: 7.2769
Epoch 1/3 | Step 400/3164 | Train 

In [7]:
@torch.no_grad()
def generate_summary(
    model, 
    tokenizer, 
    encoder_input_ids, 
    max_new_tokens=300, 
    min_new_tokens=120, 
    temperature=0.7, 
    top_p=0.9, 
    repetition_penalty=1.15,
    device="cuda"
):
    """
    Sequence-to-Sequence autoregressive decoding for legal document summarization.
    """
    model.eval()
    enc_ids = encoder_input_ids.unsqueeze(0).to(device) if encoder_input_ids.dim() == 1 else encoder_input_ids.to(device)
    enc_mask = (enc_ids != tokenizer.pad_token_id).long()
    
    # Encode document representation once
    encoder_hidden_states = model.model.encode(input_ids=enc_ids, attention_mask=enc_mask)
    
    # Initialize decoder with BOS / PAD token
    dec_input_ids = torch.tensor([[tokenizer.pad_token_id]], dtype=torch.long, device=device)
    
    for step in range(max_new_tokens):
        decoder_hidden_states = model.model.decode(
            decoder_input_ids=dec_input_ids,
            encoder_hidden_states=encoder_hidden_states,
            encoder_attention_mask=enc_mask
        )
        
        logits = model.lm_head(decoder_hidden_states)
        next_token_logits = logits[:, -1, :].clone() / max(temperature, 1e-5)
        
        # Apply repetition penalty
        if repetition_penalty != 1.0:
            for token_id in set(dec_input_ids[0].cpu().tolist()):
                if next_token_logits[0, token_id] < 0:
                    next_token_logits[0, token_id] *= repetition_penalty
                else:
                    next_token_logits[0, token_id] /= repetition_penalty
                    
        # Suppress EOS token until min_new_tokens is reached
        if step < min_new_tokens and tokenizer.eos_token_id is not None:
            next_token_logits[:, tokenizer.eos_token_id] = float("-inf")
            
        # Top-p (Nucleus) Filtering
        sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        
        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = 0
        
        indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
        next_token_logits[indices_to_remove] = float("-inf")
        
        probs = F.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        
        dec_input_ids = torch.cat([dec_input_ids, next_token], dim=1)
        
        if step >= min_new_tokens and next_token.item() == tokenizer.eos_token_id:
            break
            
    gen_tokens = dec_input_ids[0, 1:].cpu().tolist()  # Exclude start token
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

print("Seq2Seq Autoregressive generation helper configured successfully.")

Seq2Seq Autoregressive generation helper configured successfully.


In [8]:
print("Reloading Best Model Checkpoint for Phase 6 Submission Generation...")

if os.path.exists(BEST_CHECKPOINT_PATH):
    checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint["trainable_state_dict"], strict=False)
    print(f"Successfully reloaded checkpoint from Epoch {checkpoint['epoch']} (Best Val Loss: {checkpoint['best_val_loss']:.4f})\n")

test_dataset = Seq2SeqSummarizationDataset(test_data, tokenizer, max_doc_len=CFG["max_doc_len"], is_test=True)

model.eval()
predictions = []

print(f"Generating Abstractive Summaries for {len(test_dataset)} Test Documents...")

for idx, sample in enumerate(test_dataset):
    doc_id = sample["id"]
    enc_input_ids = torch.tensor(sample["encoder_input_ids"], dtype=torch.long)
    
    generated_text = generate_summary(
        model=model,
        tokenizer=tokenizer,
        encoder_input_ids=enc_input_ids,
        max_new_tokens=CFG["gen_max_new_tokens"],
        min_new_tokens=CFG["gen_min_new_tokens"],
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.15,
        device=device
    )
    
    if not generated_text:
        generated_text = "The legal document provides rulings and judicial findings on the case."
        
    predictions.append({
        "id": doc_id,
        "summary": generated_text
    })
    
    if (idx + 1) % CFG["log_interval"] == 0 or (idx + 1) == len(test_dataset):
        print(f"Generated {idx+1}/{len(test_dataset)} summaries.")

# Serialization and zip packaging
output_dir = "./phase6_enc_dec_qlora_summ"
os.makedirs(output_dir, exist_ok=True)
pred_path = os.path.join(output_dir, "prediction.json")
sub_zip_path = os.path.join(output_dir, "submission.zip")

with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(sub_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(pred_path, arcname="prediction.json")

print(f"\nPhase 6 Encoder-Decoder QLoRA Summarization Complete! Generated {len(predictions)} summaries.")
print(f"Submission saved to: {sub_zip_path}")

if DEBUG:
    print("\n[DEBUG] Sample Abstractive Summary Output:")
    print(json.dumps(predictions[:2], indent=2))

Reloading Best Model Checkpoint for Phase 6 Submission Generation...
Successfully reloaded checkpoint from Epoch 3 (Best Val Loss: 5.3073)

Generating Abstractive Summaries for 100 Test Documents...
Generated 20/100 summaries.
Generated 40/100 summaries.
Generated 60/100 summaries.
Generated 80/100 summaries.
Generated 100/100 summaries.

Phase 6 Encoder-Decoder QLoRA Summarization Complete! Generated 100 summaries.
Submission saved to: ./phase6_enc_dec_qlora_summ/submission.zip
